# 🌿 Sage · Memory Masterclass — Part 2: Managing Memory

*Keeping memory healthy, durable, and production-ready.*

Sections: **L5** working memory (compaction/rewind) · **L6** surviving time (durability) · **L7** managed cloud. Run the Setup and Sage cells first.

### ⚙️ Setup

In [ ]:
# @title ⚙️ Setup — install ADK and add your Gemini API key  { display-mode: "form" }
!pip install -q "google-adk[db]==2.3.0" aiosqlite greenlet
import os
try:
    from google.colab import userdata          # Colab Secrets (recommended)
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini API key: ")
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"
print("✅ ready")

### 🌿 The shared Sage base (run this once)

In [ ]:
# @title 🌿 Meet Sage — the shared base (tools + canned recipes + a tiny run helper)
# The *tools* don't change from level to level; only the MEMORY WIRING around Sage does.
from google.adk.agents import Agent
from google.adk.tools import ToolContext
from google.genai import types

MODEL = "gemini-3-flash-preview"

RECIPES = [
    {"name": "lentil curry",     "tags": ["vegetarian","budget","hearty"], "minutes": 30, "contains": []},
    {"name": "black bean tacos", "tags": ["vegetarian","budget","quick"],  "minutes": 20, "contains": []},
    {"name": "mushroom risotto", "tags": ["vegetarian","cozy"],            "minutes": 40, "contains": ["mushroom"]},
    {"name": "grilled salmon bowl","tags": ["pescatarian","healthy"],      "minutes": 25, "contains": ["fish"]},
    {"name": "chicken fajitas",  "tags": ["quick","budget"],               "minutes": 25, "contains": ["chicken"]},
]

def pick(preferences):
    prefs = " ".join(preferences).lower()
    for r in RECIPES:
        if "vegetarian" in prefs and "vegetarian" not in r["tags"]: continue
        if "quick" in prefs and "quick" not in r["tags"]: continue
        if any(bad in prefs for bad in r["contains"]): continue
        return r
    return RECIPES[0]

def note_preference(preference: str, tool_context: ToolContext) -> dict:
    """Save a food preference or restriction (e.g. 'vegetarian', 'no mushrooms')."""
    prefs = list(tool_context.state.get("preferences", []))
    if preference not in prefs: prefs.append(preference)
    tool_context.state["preferences"] = prefs
    return {"status": "saved", "preferences": prefs}

def suggest_meal(tool_context: ToolContext) -> dict:
    """Suggest a meal that respects the user's saved preferences."""
    prefs = tool_context.state.get("preferences", [])
    r = pick(prefs)
    return {"suggestion": r["name"], "minutes": r["minutes"], "considering": prefs}

SAGE_INSTRUCTION = (
    "You are Sage, a warm, concise meal-planning assistant. "
    "When the user shares a food preference, call note_preference. "
    "When they ask what to eat, call suggest_meal. Keep replies to 1-2 sentences."
)

def build_sage(tools=None, **kw):
    return Agent(name="sage", model=MODEL,
                 instruction=kw.pop("instruction", SAGE_INSTRUCTION),
                 tools=tools if tools is not None else [note_preference, suggest_meal], **kw)

async def say(runner, session_id, text, user_id="alice"):
    reply = ""
    async for ev in runner.run_async(user_id=user_id, session_id=session_id,
        new_message=types.Content(role="user", parts=[types.Part(text=text)])):
        for f in ev.get_function_calls() or []:
            print(f"       · (Sage used {f.name}({dict(f.args)}))")
        if ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.text: reply += p.text
    print(f"  🧑 {text}\n  🌿 {reply.strip()}\n")
    return reply.strip()

print("🌿 Sage is ready")

## L5 · Working memory — compaction & rewind 🗜️↩️

The context window is Sage's *short-term* working memory. Two tools keep it healthy:
**compaction** summarizes long chats so they stay affordable, and **rewind** undoes
a turn. (A third, context *caching*, is a transparent cost optimization — nothing
to watch.)

In [ ]:
from google.adk.agents import Agent
from google.adk.apps.app import App, EventsCompactionConfig
from google.adk.apps.llm_event_summarizer import LlmEventSummarizer
from google.adk.models import Gemini
from google.adk.runners import InMemoryRunner, Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import ToolContext

print("========== COMPACTION ==========")
chat = Agent(name="sage", model=MODEL, instruction="You are Sage. Answer in ONE short sentence.")
app = App(name="sage", root_agent=chat, events_compaction_config=EventsCompactionConfig(
    compaction_interval=3, overlap_size=1, summarizer=LlmEventSummarizer(llm=Gemini(model=MODEL))))
ss = InMemorySessionService(); runner = Runner(app=app, session_service=ss)
await ss.create_session(app_name="sage", user_id="alice", session_id="s")
for i in range(6):
    async for _ in runner.run_async(user_id="alice", session_id="s",
        new_message=types.Content(role="user", parts=[types.Part(text=f"Quick dinner idea #{i+1}?")])): pass
s = await ss.get_session(app_name="sage", user_id="alice", session_id="s")
comp = [e for e in s.events if getattr(getattr(e,'actions',None),'compaction',None)]
print(f"  {len(s.events)} events; {len(comp)} compaction summary(ies).")
if comp:
    c = comp[0].actions.compaction; content = getattr(c,'compacted_content',c)
    print("  📝", " ".join(p.text for p in content.parts if getattr(p,'text',None))[:180], "…")

print("\\n========== REWIND ==========")
def set_plan(dish: str, tool_context: ToolContext) -> dict:
    """Set tonight's dinner plan."""
    tool_context.state["plan"] = dish; return {"plan": dish}
sage = Agent(name="sage", model=MODEL, tools=[set_plan],
             instruction="When the user names a dinner dish, call set_plan. Be brief.")
r = InMemoryRunner(agent=sage, app_name="sage"); ss2 = r.session_service
await ss2.create_session(app_name="sage", user_id="alice", session_id="s")
async for _ in r.run_async(user_id="alice", session_id="s",
    new_message=types.Content(role="user", parts=[types.Part(text="Plan lentil curry tonight.")])): pass
inv = None
async for ev in r.run_async(user_id="alice", session_id="s",
    new_message=types.Content(role="user", parts=[types.Part(text="Actually, change it to mushroom soup.")])): inv = ev.invocation_id
print("  plan =", (await ss2.get_session(app_name='sage',user_id='alice',session_id='s')).state.get("plan"))
await r.rewind_async(user_id="alice", session_id="s", rewind_before_invocation_id=inv)
print("  after rewind =", (await ss2.get_session(app_name='sage',user_id='alice',session_id='s')).state.get("plan"), " ✅ undone")

## L6 · Surviving time — durability ⏳

Where memory meets long-running work. Sage's grocery order is a
`LongRunningFunctionTool`: it pauses for confirmation, the run **ends**, and it
resumes later when the user confirms. *(Full crash-and-restart story: the
Long-Running Agent lab — github.com/cuppibla/loop-lab-onboarding.)*

In [ ]:
from google.adk.agents import Agent
from google.adk.apps.app import App, ResumabilityConfig
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import LongRunningFunctionTool, ToolContext

def order_groceries(items: str, tool_context: ToolContext) -> dict:
    """Place a grocery order. Needs the user to confirm first."""
    return {"status": "pending_confirmation", "items": items}

sage = Agent(name="sage", model=MODEL, tools=[LongRunningFunctionTool(order_groceries)],
    instruction="When asked to order groceries, call order_groceries. After you receive "
                "confirmed=true, say the order is placed. Be brief.")
app = App(name="sage", root_agent=sage, resumability_config=ResumabilityConfig(is_resumable=True))
ss = InMemorySessionService(); runner = Runner(app=app, session_service=ss)
await ss.create_session(app_name="sage", user_id="alice", session_id="s")

async def run(message):
    pend = None
    async for ev in runner.run_async(user_id="alice", session_id="s", new_message=message):
        for f in ev.get_function_calls() or []:
            if ev.long_running_tool_ids and f.id in ev.long_running_tool_ids:
                pend = (f.id, f.name); print(f"       · {f.name} → PENDING (awaiting confirmation)")
        if ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.text and p.text.strip(): print(f"  🌿 {p.text.strip()}")
    return pend

print("── Sage places an order — it pauses, then the run ends ──")
pend = await run(types.Content(role="user", parts=[types.Part(text="Order groceries: lentils, onions, coconut milk.")]))
print("── Later — the user confirms → the run resumes and completes ──")
fid, fname = pend
await run(types.Content(role="user", parts=[types.Part(
    function_response=types.FunctionResponse(id=fid, name=fname, response={"confirmed": True}))]))
print("✅ A long-running task paused, the run ended, and it resumed later on confirmation.")

## L7 · Managed in the cloud — Vertex sessions + Memory Bank ☁️

The payoff of the whole hierarchy: **the Sage code doesn't change to go to
production — only the services swap.**

```python
# local (L2/L3)                 # managed (L7)
DatabaseSessionService   →  VertexAiSessionService(project, location, agent_engine_id)
InMemoryMemoryService    →  VertexAiMemoryBankService(project, location, agent_engine_id)
```

You get managed sessions + Memory Bank with nothing to run or back up. This step
needs a GCP project + an Agent Engine instance; everything above runs with no cloud.

In [ ]:
# Wiring only (runs the live version if you've set GOOGLE_CLOUD_PROJECT + AGENT_ENGINE_ID).
import os
project = os.environ.get("GOOGLE_CLOUD_PROJECT"); aeid = os.environ.get("AGENT_ENGINE_ID")
if not (project and aeid):
    print("⏭ Set GOOGLE_GENAI_USE_VERTEXAI=True, GOOGLE_CLOUD_PROJECT, GOOGLE_CLOUD_LOCATION,")
    print("  and AGENT_ENGINE_ID (an Agent Engine instance) to run Sage on managed services.")
else:
    from google.adk.sessions import VertexAiSessionService
    from google.adk.memory import VertexAiMemoryBankService
    from google.adk.runners import Runner
    loc = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")
    runner = Runner(agent=build_sage(), app_name="sage",
        session_service=VertexAiSessionService(project=project, location=loc, agent_engine_id=aeid),
        memory_service=VertexAiMemoryBankService(project=project, location=loc, agent_engine_id=aeid))
    await runner.session_service.create_session(app_name="sage", user_id="alice", session_id="s")
    await say(runner, "s", "Hi Sage, I'm vegetarian — remember that.")
    print("✅ Same Sage, now on managed Vertex sessions + Memory Bank.")

## Recap — the Memory Hierarchy

| Rung | Concept | ADK piece |
|---|---|---|
| within a turn | stateless | `Runner` + `InMemorySessionService` |
| within a chat | scratchpad | `session.state` |
| across chats | durable + scoped | `DatabaseSessionService` · `user:`/`app:` |
| long-term | searchable memory | `MemoryService` · `add_session_to_memory` / `search_memory` |
| files | artifacts | `save/load_artifact` |
| working memory | compaction · rewind · caching | `EventsCompactionConfig` · `rewind_async` |
| surviving time | durability | `ResumabilityConfig` · `LongRunningFunctionTool` |
| managed | zero-infra | `VertexAiSessionService` + `VertexAiMemoryBankService` |

**"Persistence is not memory."** Pick the rung your agent actually needs.